In [1]:
"""
Visualization — All 7 Pruning Methods Comparison
=================================================
Reads all 7 pruning JSON files and produces:
  - Per-method plots (accuracy, compression, inference)
  - Cross-method comparison plots

Expected JSON files (run the pruning scripts first):
  1_unstructured_Pruning.json
  2_structured_Pruning.json
  3_magnitude_Pruning.json
  4_movement_Pruning.json
  5_lottery_ticket_Pruning.json
  6_iterative_Pruning.json
  7_oneshot_Pruning.json

Output PNGs saved in ./plots/
"""

import json
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

warnings.filterwarnings("ignore")

# ── Style ─────────────────────────────────────────────────────────────────────
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "font.family"       : "sans-serif",
    "font.sans-serif"   : ["Arial", "DejaVu Sans"],
    "axes.linewidth"    : 0.8,
    "grid.alpha"        : 0.3,
    "axes.spines.top"   : False,
    "axes.spines.right" : False,
})

COLORS = {
    "unstructured"    : "#2E86AB",
    "structured"      : "#A23B72",
    "magnitude"       : "#F18F01",
    "movement"        : "#4CAF50",
    "lottery_ticket"  : "#E91E63",
    "iterative"       : "#9C27B0",
    "oneshot"         : "#FF5722",
    "baseline"        : "#6C757D",
    "threshold"       : "#F44336",
    "l1"              : "#2196F3",
    "l2"              : "#FF9800",
    "random"          : "#9E9E9E",
}

PLOTS_DIR = Path("plots")
PLOTS_DIR.mkdir(exist_ok=True)

STANDARD_SPARSITIES = [0.30, 0.50, 0.70, 0.80, 0.90]


# ── Helpers ───────────────────────────────────────────────────────────────────
def savefig(name, tight=True):
    path = PLOTS_DIR / name
    if tight:
        plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.close()
    print(f"  ✓ {path}")


def load_json(path):
    if not os.path.exists(path):
        print(f"  WARNING: {path} not found, skipping.")
        return None
    with open(path) as f:
        return json.load(f)


def get_baseline_acc(data):
    return data.get("baseline", {}).get("accuracy", 0.927)


def add_baseline_line(ax, data, label=True, color=None):
    acc = get_baseline_acc(data) * 100
    c   = color or COLORS["baseline"]
    ax.axhline(acc, color=c, linestyle=":", linewidth=2,
               label=f"Baseline ({acc:.2f}%)" if label else None)


def fmt_sp(sp):
    return f"{int(sp*100)}%"


# ══════════════════════════════════════════════════════════════════════════════
# 1. UNSTRUCTURED PRUNING
# ══════════════════════════════════════════════════════════════════════════════
def plot_unstructured(data):
    if data is None: return
    df = pd.DataFrame(data["results"])
    bl = data["baseline"]

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle("1. Unstructured Pruning — Global L1", fontsize=13, fontweight="bold")
    sp = df["actual_sparsity"].values

    # Accuracy
    axes[0].plot(sp, df["accuracy"]*100, "o-", color=COLORS["unstructured"], lw=2.5, ms=8, label="Accuracy")
    axes[0].plot(sp, df["f1"]*100, "s--", color=COLORS["movement"], lw=2, ms=7, label="F1")
    axes[0].axhline(bl["accuracy"]*100, color=COLORS["baseline"], ls=":", lw=2, label="Baseline")
    axes[0].set_xlabel("Sparsity"); axes[0].set_ylabel("Score (%)"); axes[0].set_title("Accuracy & F1")
    axes[0].set_xticks(sp); axes[0].set_xticklabels([fmt_sp(s) for s in sp])
    axes[0].legend()

    # Compression
    axes[1].bar(sp*100, df["params_active"]/1e6, width=7, color=COLORS["unstructured"], alpha=0.8)
    axes[1].axhline(bl["params"]/1e6, color=COLORS["threshold"], ls="--", lw=2, label="Baseline")
    axes[1].set_xlabel("Sparsity"); axes[1].set_ylabel("Active Params (M)"); axes[1].set_title("Model Compression")
    axes[1].set_xticks(sp*100); axes[1].set_xticklabels([fmt_sp(s) for s in sp])
    axes[1].legend()

    # Size comparison
    x = np.arange(len(sp))
    w = 0.35
    axes[2].bar(x - w/2, df["size_sparse_theoretical_mb"], width=w, label="Sparse (theoretical)", color=COLORS["unstructured"], alpha=0.85)
    axes[2].bar(x + w/2, df["size_disk_mb"], width=w, label="Disk .pth", color=COLORS["magnitude"], alpha=0.85)
    axes[2].set_xticks(x); axes[2].set_xticklabels([fmt_sp(s) for s in sp])
    axes[2].set_xlabel("Sparsity"); axes[2].set_ylabel("Size (MB)"); axes[2].set_title("Size Breakdown")
    axes[2].legend()

    savefig("1_unstructured_pruning.png")


# ══════════════════════════════════════════════════════════════════════════════
# 2. STRUCTURED PRUNING
# ══════════════════════════════════════════════════════════════════════════════
def plot_structured(data):
    if data is None: return
    df = pd.DataFrame(data["results"])
    bl = data["baseline"]

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle("2. Structured Pruning — L1 Filter Pruning", fontsize=13, fontweight="bold")
    ratio = df["filter_pruning_ratio"].values

    axes[0].plot(ratio, df["accuracy"]*100, "o-", color=COLORS["structured"], lw=2.5, ms=8, label="Accuracy")
    axes[0].plot(ratio, df["f1"]*100, "s--", color=COLORS["movement"], lw=2, ms=7, label="F1")
    axes[0].axhline(bl["accuracy"]*100, color=COLORS["baseline"], ls=":", lw=2, label="Baseline")
    axes[0].set_xlabel("Filter Pruning Ratio"); axes[0].set_ylabel("Score (%)"); axes[0].set_title("Accuracy & F1")
    axes[0].set_xticks(ratio); axes[0].set_xticklabels([fmt_sp(r) for r in ratio])
    axes[0].legend()

    axes[1].plot(ratio, df["structured_sparsity"]*100, "o-", color=COLORS["structured"], lw=2.5, ms=8, label="Filter sparsity")
    axes[1].plot(ratio, df["weight_sparsity"]*100, "s--", color=COLORS["magnitude"], lw=2, ms=7, label="Weight sparsity")
    axes[1].set_xlabel("Filter Pruning Ratio"); axes[1].set_ylabel("Sparsity (%)"); axes[1].set_title("Sparsity Breakdown")
    axes[1].set_xticks(ratio); axes[1].set_xticklabels([fmt_sp(r) for r in ratio])
    axes[1].legend()

    drop_colors = [COLORS["movement"] if d <= 0 else COLORS["threshold"] for d in df["accuracy_drop"]]
    axes[2].bar(ratio*100, df["accuracy_drop"]*100, width=7, color=drop_colors, alpha=0.85)
    axes[2].axhline(0, color="black", lw=0.8)
    axes[2].set_xlabel("Filter Pruning Ratio"); axes[2].set_ylabel("Accuracy Drop (%)"); axes[2].set_title("Accuracy Trade-off")
    axes[2].set_xticks(ratio*100); axes[2].set_xticklabels([fmt_sp(r) for r in ratio])

    savefig("2_structured_pruning.png")


# ══════════════════════════════════════════════════════════════════════════════
# 3. MAGNITUDE PRUNING
# ══════════════════════════════════════════════════════════════════════════════
def plot_magnitude(data):
    if data is None: return
    df  = pd.DataFrame(data["results"])
    bl  = data["baseline"]

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle("3. Magnitude Pruning — L1-local vs L2-local vs Global-L1", fontsize=13, fontweight="bold")

    crit_styles = {
        "local_l1" : (COLORS["l1"],     "o-",  "Local L1"),
        "local_l2" : (COLORS["l2"],     "s--", "Local L2"),
        "global_l1": (COLORS["movement"],"^-.", "Global L1"),
    }

    sparsities = sorted(df["target_sparsity"].unique())
    for crit, (color, style, label) in crit_styles.items():
        sub = df[df["criterion"] == crit].sort_values("target_sparsity")
        if sub.empty: continue
        axes[0].plot(sub["actual_sparsity"], sub["accuracy"]*100, style, color=color,
                     lw=2.5, ms=8, label=label)
        axes[1].plot(sub["actual_sparsity"], sub["accuracy_drop"]*100, style, color=color,
                     lw=2.5, ms=8, label=label)

    axes[0].axhline(bl["accuracy"]*100, color=COLORS["baseline"], ls=":", lw=2, label="Baseline")
    axes[0].set_xlabel("Sparsity"); axes[0].set_ylabel("Accuracy (%)"); axes[0].set_title("Accuracy by Criterion")
    axes[0].set_xticks(sparsities); axes[0].set_xticklabels([fmt_sp(s) for s in sparsities])
    axes[0].legend()

    axes[1].axhline(0, color="black", lw=0.8)
    axes[1].axhline(data["max_acc_drop_threshold"]*100, color=COLORS["threshold"], ls="--", lw=2, label="Threshold")
    axes[1].set_xlabel("Sparsity"); axes[1].set_ylabel("Accuracy Drop (%)"); axes[1].set_title("Accuracy Drop by Criterion")
    axes[1].set_xticks(sparsities); axes[1].set_xticklabels([fmt_sp(s) for s in sparsities])
    axes[1].legend()

    savefig("3_magnitude_pruning.png")


# ══════════════════════════════════════════════════════════════════════════════
# 4. MOVEMENT PRUNING
# ══════════════════════════════════════════════════════════════════════════════
def plot_movement(data):
    if data is None: return
    df = pd.DataFrame(data["results"])
    bl = data["baseline"]

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle("4. Movement Pruning — Taylor |w·grad| Importance", fontsize=13, fontweight="bold")
    sp = df["actual_sparsity"].values

    axes[0].plot(sp, df["accuracy"]*100, "o-", color=COLORS["movement"], lw=2.5, ms=8, label="Accuracy")
    axes[0].plot(sp, df["f1"]*100, "s--", color=COLORS["unstructured"], lw=2, ms=7, label="F1")
    axes[0].axhline(bl["accuracy"]*100, color=COLORS["baseline"], ls=":", lw=2, label="Baseline")
    axes[0].set_xlabel("Sparsity"); axes[0].set_ylabel("Score (%)"); axes[0].set_title("Accuracy & F1")
    axes[0].set_xticks(sp); axes[0].set_xticklabels([fmt_sp(s) for s in sp])
    axes[0].legend()

    axes[1].bar(sp*100, df["disk_saved_mb"], width=7, color=COLORS["movement"], alpha=0.85)
    for bar, val in zip(axes[1].patches, df["disk_saved_mb"]):
        axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1,
                     f"{val:.1f}", ha="center", va="bottom", fontsize=9)
    axes[1].set_xlabel("Sparsity"); axes[1].set_ylabel("Disk Saved (MB)"); axes[1].set_title("Disk Compression")
    axes[1].set_xticks(sp*100); axes[1].set_xticklabels([fmt_sp(s) for s in sp])

    savefig("4_movement_pruning.png")


# ══════════════════════════════════════════════════════════════════════════════
# 5. LOTTERY TICKET
# ══════════════════════════════════════════════════════════════════════════════
def plot_lottery_ticket(data):
    if data is None: return
    df = pd.DataFrame(data["results"])
    bl = data["baseline"]

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle("5. Lottery Ticket Hypothesis — Winning Ticket vs Random Init", fontsize=13, fontweight="bold")
    sp = df["actual_sparsity"].values

    axes[0].plot(sp, df["winning_ticket_accuracy"]*100, "o-", color=COLORS["lottery_ticket"],
                 lw=2.5, ms=9, label="Winning Ticket (trained weights)")
    axes[0].plot(sp, df["random_init_accuracy"]*100, "s--", color=COLORS["random"],
                 lw=2, ms=8, label="Random Init (same mask)")
    axes[0].axhline(bl["accuracy"]*100, color=COLORS["baseline"], ls=":", lw=2, label="Full Baseline")
    axes[0].fill_between(sp, df["random_init_accuracy"]*100, df["winning_ticket_accuracy"]*100,
                          alpha=0.15, color=COLORS["lottery_ticket"], label="LTH Advantage")
    axes[0].set_xlabel("Sparsity"); axes[0].set_ylabel("Accuracy (%)"); axes[0].set_title("LTH: Trained vs Random")
    axes[0].set_xticks(sp); axes[0].set_xticklabels([fmt_sp(s) for s in sp])
    axes[0].legend(fontsize=9)

    axes[1].bar(sp*100, df["lth_advantage_pct"], width=7, color=COLORS["lottery_ticket"], alpha=0.85)
    axes[1].axhline(0, color="black", lw=0.8)
    for bar, val in zip(axes[1].patches, df["lth_advantage_pct"]):
        axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
                     f"{val:.2f}%", ha="center", va="bottom", fontsize=9)
    axes[1].set_xlabel("Sparsity"); axes[1].set_ylabel("LTH Advantage (%)"); axes[1].set_title("LTH Advantage (trained - random)")
    axes[1].set_xticks(sp*100); axes[1].set_xticklabels([fmt_sp(s) for s in sp])

    savefig("5_lottery_ticket_pruning.png")


# ══════════════════════════════════════════════════════════════════════════════
# 6. ITERATIVE PRUNING
# ══════════════════════════════════════════════════════════════════════════════
def plot_iterative(data):
    if data is None: return
    traj = pd.DataFrame(data["trajectory"])
    bl   = data["baseline"]

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle("6. Iterative Pruning — Full Trajectory", fontsize=13, fontweight="bold")

    sp = traj["cumulative_sparsity"].values

    axes[0].plot(sp*100, traj["accuracy"]*100, "o-", color=COLORS["iterative"], lw=2.5, ms=6)
    axes[0].axhline(bl["accuracy"]*100, color=COLORS["baseline"], ls=":", lw=2, label="Baseline")
    axes[0].axhline((bl["accuracy"]-data["max_acc_drop_threshold"])*100,
                    color=COLORS["threshold"], ls="--", lw=2, label="Threshold")
    axes[0].set_xlabel("Cumulative Sparsity (%)"); axes[0].set_ylabel("Accuracy (%)"); axes[0].set_title("Accuracy Trajectory")
    axes[0].legend()

    axes[1].plot(traj["round"], traj["accuracy"]*100, "o-", color=COLORS["iterative"], lw=2.5, ms=6)
    axes[1].axhline(bl["accuracy"]*100, color=COLORS["baseline"], ls=":", lw=2, label="Baseline")
    axes[1].set_xlabel("Pruning Round"); axes[1].set_ylabel("Accuracy (%)"); axes[1].set_title("Accuracy per Round")
    axes[1].legend()

    axes[2].plot(sp*100, traj["size_sparse_theoretical_mb"], "o-", color=COLORS["iterative"], lw=2.5, ms=6)
    axes[2].set_xlabel("Cumulative Sparsity (%)"); axes[2].set_ylabel("Sparse Size (MB)"); axes[2].set_title("Size Reduction Trajectory")

    savefig("6_iterative_pruning.png")


# ══════════════════════════════════════════════════════════════════════════════
# 7. ONE-SHOT PRUNING
# ══════════════════════════════════════════════════════════════════════════════
def plot_oneshot(data):
    if data is None: return
    df = pd.DataFrame(data["results"])
    bl = data["baseline"]

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle("7. One-Shot Pruning — L1 vs L2 vs Random", fontsize=13, fontweight="bold")

    variant_styles = {
        "oneshot_l1_global": (COLORS["l1"],     "o-",  "L1 Global"),
        "oneshot_l2_global": (COLORS["l2"],     "s--", "L2 Global"),
        "oneshot_random"   : (COLORS["random"], "^:",  "Random"),
    }

    sparsities = sorted(df["target_sparsity"].unique())
    for var, (color, style, label) in variant_styles.items():
        sub = df[df["variant"] == var].sort_values("target_sparsity")
        if sub.empty: continue
        axes[0].plot(sub["actual_sparsity"], sub["accuracy"]*100, style, color=color,
                     lw=2.5, ms=8, label=label)
        axes[1].plot(sub["actual_sparsity"], sub["accuracy_drop"]*100, style, color=color,
                     lw=2.5, ms=8, label=label)

    axes[0].axhline(bl["accuracy"]*100, color=COLORS["baseline"], ls=":", lw=2, label="Baseline")
    axes[0].set_xlabel("Sparsity"); axes[0].set_ylabel("Accuracy (%)"); axes[0].set_title("Accuracy by Variant")
    axes[0].set_xticks(sparsities); axes[0].set_xticklabels([fmt_sp(s) for s in sparsities])
    axes[0].legend()

    axes[1].axhline(0, color="black", lw=0.8)
    axes[1].axhline(data["max_acc_drop_threshold"]*100, color=COLORS["threshold"], ls="--", lw=2, label="Max Threshold")
    axes[1].set_xlabel("Sparsity"); axes[1].set_ylabel("Accuracy Drop (%)"); axes[1].set_title("Accuracy Drop by Variant")
    axes[1].set_xticks(sparsities); axes[1].set_xticklabels([fmt_sp(s) for s in sparsities])
    axes[1].legend()

    savefig("7_oneshot_pruning.png")


# ══════════════════════════════════════════════════════════════════════════════
# CROSS-METHOD COMPARISON
# ══════════════════════════════════════════════════════════════════════════════

def extract_for_comparison(all_data):
    """
    Extract the best (highest accuracy within drop threshold) result
    per method at each standard sparsity level, for cross-method comparison.
    Returns list of dicts: {method, sparsity, accuracy, acc_drop, sparse_mb, disk_mb}
    """
    rows = []

    def nearest(lst, target):
        return min(lst, key=lambda x: abs(x.get("actual_sparsity", x.get("cumulative_sparsity", 0)) - target))

    # 1. Unstructured
    d = all_data.get("unstructured")
    if d:
        for r in d["results"]:
            rows.append({"method": "Unstructured", "sparsity": r["actual_sparsity"],
                         "accuracy": r["accuracy"], "acc_drop": r["accuracy_drop"],
                         "sparse_mb": r.get("size_sparse_theoretical_mb", 0),
                         "disk_mb": r.get("size_disk_mb", 0)})

    # 2. Structured (use weight_sparsity for x-axis comparison)
    d = all_data.get("structured")
    if d:
        for r in d["results"]:
            rows.append({"method": "Structured", "sparsity": r.get("weight_sparsity", r.get("structured_sparsity", 0)),
                         "accuracy": r["accuracy"], "acc_drop": r["accuracy_drop"],
                         "sparse_mb": r.get("size_sparse_theoretical_mb", 0),
                         "disk_mb": r.get("size_disk_mb", 0)})

    # 3. Magnitude — use global_l1 for fair comparison
    d = all_data.get("magnitude")
    if d:
        df = pd.DataFrame(d["results"])
        for _, r in df[df["criterion"] == "global_l1"].iterrows():
            rows.append({"method": "Magnitude(global-L1)", "sparsity": r["actual_sparsity"],
                         "accuracy": r["accuracy"], "acc_drop": r["accuracy_drop"],
                         "sparse_mb": r.get("size_sparse_theoretical_mb", 0),
                         "disk_mb": r.get("size_disk_mb", 0)})

    # 4. Movement
    d = all_data.get("movement")
    if d:
        for r in d["results"]:
            rows.append({"method": "Movement", "sparsity": r["actual_sparsity"],
                         "accuracy": r["accuracy"], "acc_drop": r["accuracy_drop"],
                         "sparse_mb": r.get("size_sparse_theoretical_mb", 0),
                         "disk_mb": r.get("size_disk_mb", 0)})

    # 5. Lottery ticket (winning ticket)
    d = all_data.get("lottery")
    if d:
        for r in d["results"]:
            rows.append({"method": "LotteryTicket", "sparsity": r["actual_sparsity"],
                         "accuracy": r["winning_ticket_accuracy"], "acc_drop": r["winning_ticket_accuracy_drop"],
                         "sparse_mb": r.get("size_sparse_theoretical_mb", 0),
                         "disk_mb": r.get("size_disk_mb", 0)})

    # 6. Iterative — use checkpointed results
    d = all_data.get("iterative")
    if d:
        for r in d.get("checkpointed_results", d["trajectory"]):
            sp_key = "cumulative_sparsity" if "cumulative_sparsity" in r else "actual_sparsity"
            rows.append({"method": "Iterative", "sparsity": r[sp_key],
                         "accuracy": r["accuracy"], "acc_drop": r["accuracy_drop"],
                         "sparse_mb": r.get("size_sparse_theoretical_mb", 0),
                         "disk_mb": r.get("size_disk_mb", 0)})

    # 7. One-shot — use l1_global variant
    d = all_data.get("oneshot")
    if d:
        df = pd.DataFrame(d["results"])
        for _, r in df[df["variant"] == "oneshot_l1_global"].iterrows():
            rows.append({"method": "OneShot(L1)", "sparsity": r["actual_sparsity"],
                         "accuracy": r["accuracy"], "acc_drop": r["accuracy_drop"],
                         "sparse_mb": r.get("size_sparse_theoretical_mb", 0),
                         "disk_mb": r.get("size_disk_mb", 0)})

    return pd.DataFrame(rows)


def plot_comparison_accuracy(comp_df, baseline_acc):
    """All methods: accuracy vs sparsity."""
    if comp_df.empty: return
    fig, ax = plt.subplots(figsize=(13, 7))
    methods = comp_df["method"].unique()
    method_colors = {m: list(COLORS.values())[i % len(COLORS)] for i, m in enumerate(methods)}

    for method in methods:
        sub = comp_df[comp_df["method"] == method].sort_values("sparsity")
        ax.plot(sub["sparsity"]*100, sub["accuracy"]*100, "o-", lw=2.5, ms=8,
                label=method, color=method_colors[method])

    ax.axhline(baseline_acc*100, color=COLORS["baseline"], ls=":", lw=2.5, label="Baseline")
    ax.set_xlabel("Sparsity (%)", fontsize=12, fontweight="bold")
    ax.set_ylabel("Accuracy (%)", fontsize=12, fontweight="bold")
    ax.set_title("All Pruning Methods: Accuracy vs Sparsity", fontsize=14, fontweight="bold")
    ax.legend(fontsize=9, loc="lower left")
    savefig("comparison_accuracy.png")


def plot_comparison_acc_drop(comp_df, threshold):
    if comp_df.empty: return
    fig, ax = plt.subplots(figsize=(13, 7))
    methods = comp_df["method"].unique()
    method_colors = {m: list(COLORS.values())[i % len(COLORS)] for i, m in enumerate(methods)}

    for method in methods:
        sub = comp_df[comp_df["method"] == method].sort_values("sparsity")
        ax.plot(sub["sparsity"]*100, sub["acc_drop"]*100, "o-", lw=2.5, ms=8,
                label=method, color=method_colors[method])

    ax.axhline(0, color="black", lw=0.8)
    ax.axhline(threshold*100, color=COLORS["threshold"], ls="--", lw=2.5, label=f"Max Threshold ({threshold*100:.0f}%)")
    ax.set_xlabel("Sparsity (%)", fontsize=12, fontweight="bold")
    ax.set_ylabel("Accuracy Drop (%)", fontsize=12, fontweight="bold")
    ax.set_title("All Pruning Methods: Accuracy Drop vs Sparsity", fontsize=14, fontweight="bold")
    ax.legend(fontsize=9)
    savefig("comparison_acc_drop.png")


def plot_comparison_bar_at_sparsity(comp_df, target_sparsity=0.70):
    """Bar chart comparing all methods at a fixed sparsity level."""
    if comp_df.empty: return
    methods = comp_df["method"].unique()
    method_colors = {m: list(COLORS.values())[i % len(COLORS)] for i, m in enumerate(methods)}

    accs = []
    for method in methods:
        sub = comp_df[comp_df["method"] == method].sort_values("sparsity")
        closest = sub.iloc[(sub["sparsity"] - target_sparsity).abs().argsort()[:1]]
        if not closest.empty:
            accs.append({"method": method, "accuracy": closest["accuracy"].values[0]})

    if not accs: return
    acc_df = pd.DataFrame(accs).sort_values("accuracy", ascending=True)

    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.barh(acc_df["method"], acc_df["accuracy"]*100,
                   color=[method_colors.get(m, "#888") for m in acc_df["method"]], alpha=0.85)

    for bar, val in zip(bars, acc_df["accuracy"]*100):
        ax.text(val + 0.1, bar.get_y()+bar.get_height()/2,
                f"{val:.2f}%", va="center", fontsize=10)

    ax.set_xlabel("Accuracy (%)", fontsize=11, fontweight="bold")
    ax.set_title(f"Method Comparison at ~{int(target_sparsity*100)}% Sparsity", fontsize=13, fontweight="bold")
    ax.set_xlim(85, 94)
    savefig(f"comparison_bar_{int(target_sparsity*100)}pct.png")


def plot_compression_vs_accuracy(comp_df, baseline_acc):
    """Scatter: sparse size vs accuracy — Pareto-style."""
    if comp_df.empty or "sparse_mb" not in comp_df.columns: return
    fig, ax = plt.subplots(figsize=(11, 7))
    methods = comp_df["method"].unique()
    method_colors = {m: list(COLORS.values())[i % len(COLORS)] for i, m in enumerate(methods)}

    for method in methods:
        sub = comp_df[comp_df["method"] == method]
        ax.scatter(sub["sparse_mb"], sub["accuracy"]*100,
                   color=method_colors[method], s=80, label=method, zorder=4)
        # connect points within method
        sub_sorted = sub.sort_values("sparse_mb")
        ax.plot(sub_sorted["sparse_mb"], sub_sorted["accuracy"]*100,
                color=method_colors[method], lw=1.2, alpha=0.5)

    ax.axhline(baseline_acc*100, color=COLORS["baseline"], ls=":", lw=2, label="Baseline")
    ax.set_xlabel("Theoretical Sparse Size (MB)", fontsize=11, fontweight="bold")
    ax.set_ylabel("Accuracy (%)", fontsize=11, fontweight="bold")
    ax.set_title("Compression–Accuracy Trade-off (all methods)", fontsize=13, fontweight="bold")
    ax.legend(fontsize=9, loc="lower right")
    savefig("comparison_compression_accuracy.png")


# ══════════════════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════════════════
def main():
    print(f"\n{'='*60}")
    print("  Visualization — All 7 Pruning Methods")
    print(f"{'='*60}\n")

    FILES = {
        "unstructured": "1_unstructured_Pruning.json",
        "structured"  : "2_structured_Pruning.json",
        "magnitude"   : "3_magnitude_Pruning.json",
        "movement"    : "4_movement_Pruning.json",
        "lottery"     : "5_lottery_ticket_Pruning.json",
        "iterative"   : "6_iterative_Pruning.json",
        "oneshot"     : "7_oneshot_Pruning.json",
    }

    all_data = {}
    for key, fname in FILES.items():
        d = load_json(fname)
        all_data[key] = d

    print("\nPer-method plots:")

    print("  1. Unstructured...")
    plot_unstructured(all_data["unstructured"])

    print("  2. Structured...")
    plot_structured(all_data["structured"])

    print("  3. Magnitude...")
    plot_magnitude(all_data["magnitude"])

    print("  4. Movement...")
    plot_movement(all_data["movement"])

    print("  5. Lottery Ticket...")
    plot_lottery_ticket(all_data["lottery"])

    print("  6. Iterative...")
    plot_iterative(all_data["iterative"])

    print("  7. One-Shot...")
    plot_oneshot(all_data["oneshot"])

    print("\nCross-method comparison plots:")
    comp_df = extract_for_comparison(all_data)

    # Infer baseline accuracy from any available data
    baseline_acc = 0.927
    threshold     = 0.02
    for key, d in all_data.items():
        if d and "baseline" in d:
            baseline_acc = d["baseline"].get("accuracy", 0.927)
            threshold    = d.get("max_acc_drop_threshold", 0.02)
            break

    if not comp_df.empty:
        print("  Accuracy comparison...")
        plot_comparison_accuracy(comp_df, baseline_acc)

        print("  Accuracy drop comparison...")
        plot_comparison_acc_drop(comp_df, threshold)

        print("  Bar chart at 70% sparsity...")
        plot_comparison_bar_at_sparsity(comp_df, 0.70)

        print("  Bar chart at 90% sparsity...")
        plot_comparison_bar_at_sparsity(comp_df, 0.90)

        print("  Compression vs accuracy scatter...")
        plot_compression_vs_accuracy(comp_df, baseline_acc)
    else:
        print("  Not enough data for comparison plots (run pruning scripts first).")

    print(f"\n  ✓ All plots saved in ./plots/")
    print(f"\n  Files generated:")
    for f in sorted(PLOTS_DIR.iterdir()):
        print(f"    {f}")


if __name__ == "__main__":
    main()



  Visualization — All 7 Pruning Methods


Per-method plots:
  1. Unstructured...
  ✓ plots\1_unstructured_pruning.png
  2. Structured...
  ✓ plots\2_structured_pruning.png
  3. Magnitude...
  ✓ plots\3_magnitude_pruning.png
  4. Movement...
  ✓ plots\4_movement_pruning.png
  5. Lottery Ticket...
  ✓ plots\5_lottery_ticket_pruning.png
  6. Iterative...
  ✓ plots\6_iterative_pruning.png
  7. One-Shot...
  ✓ plots\7_oneshot_pruning.png

Cross-method comparison plots:
  Accuracy comparison...
  ✓ plots\comparison_accuracy.png
  Accuracy drop comparison...
  ✓ plots\comparison_acc_drop.png
  Bar chart at 70% sparsity...
  ✓ plots\comparison_bar_70pct.png
  Bar chart at 90% sparsity...
  ✓ plots\comparison_bar_90pct.png
  Compression vs accuracy scatter...
  ✓ plots\comparison_compression_accuracy.png

  ✓ All plots saved in ./plots/

  Files generated:
    plots\1_unstructured_pruning.png
    plots\2_structured_pruning.png
    plots\3_magnitude_pruning.png
    plots\4_movement_pruning.png
